# Can Xuanwu District rooftops supply meaningful low-carbon electricity?

This case treats the rooftop photovoltaic model as a research instrument rather than as a precomputed chart. The experiment asks whether building rooftops in Xuanwu District, Nanjing, can provide a meaningful amount of renewable electricity and carbon reduction potential.

The notebook intentionally analyzes the output downloaded from the current PyGeoModel/OpenGMS task. Bundled historical result files are not used as evidence in the main workflow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import folium
from folium.plugins import HeatMap
from IPython.display import display
from pygeomodel import GeoModeler


def configure_opengms_http_timeout(default_seconds=180):
    """Let slow OpenGMS manager endpoints respond before the SDK gives up."""
    timeout_seconds = int(os.getenv("OPENGMS_LIVE_TIMEOUT_SECONDS", str(default_seconds)) or default_seconds)
    try:
        from ogmsServer2.openUtils.http_client import HttpClient
    except Exception as exc:
        print(f"OpenGMS SDK timeout helper was not applied: {exc}")
        return timeout_seconds

    if getattr(HttpClient, "_opengeolab_timeout_patch", False):
        return timeout_seconds

    original_get_sync = HttpClient.get_sync
    original_get_file_sync = HttpClient.get_file_sync
    original_post_sync = HttpClient.post_sync

    def get_sync(url, timeout=None, params=None, headers=None):
        return original_get_sync(url=url, timeout=timeout or timeout_seconds, params=params, headers=headers)

    def get_file_sync(url, timeout=None, params=None, headers=None):
        return original_get_file_sync(url=url, timeout=timeout or timeout_seconds, params=params, headers=headers)

    def post_sync(url, timeout=None, data=None, json=None, files=None, headers=None):
        return original_post_sync(url=url, timeout=timeout or timeout_seconds, data=data, json=json, files=files, headers=headers)

    HttpClient.get_sync = staticmethod(get_sync)
    HttpClient.get_file_sync = staticmethod(get_file_sync)
    HttpClient.post_sync = staticmethod(post_sync)
    HttpClient._opengeolab_timeout_patch = True
    print(f"OpenGMS SDK HTTP timeout: {timeout_seconds}s")
    return timeout_seconds


DATA_DIR = Path("data")
ROOFTOP_INVENTORY = DATA_DIR / "xuanwu" / "xuanwu.shp"
MODEL_INPUT_ZIP = DATA_DIR / "xuanwu_rooftop.zip"
MODEL_NAME = "Roof Photovoltaic Carbon Emission Reduction Potential Assessment Model"

LIVE_OUTPUT_DIR = Path("outputs/live-rooftop-pv")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = LIVE_OUTPUT_DIR / RUN_ID
DOWNLOAD_DIR = RUN_DIR / "downloads"
EXTRACT_DIR = RUN_DIR / "extracted"

for required_path in [ROOFTOP_INVENTORY, MODEL_INPUT_ZIP]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required case input: {required_path}")

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Live task artifacts for this run will be stored in: {RUN_DIR}")

## 1. Study Design

The scientific question is deliberately narrow: if the same rooftop inventory is submitted to the OpenGMS rooftop PV model today, what building-level generation potential does the live model return, and where are the high-yield rooftops concentrated?

The local shapefile is used only as the input inventory and for pre-model quality checks. The generation statistics, heatmap, and carbon estimate are derived from the fresh model output downloaded into the run directory above.

In [ ]:
rooftops = gpd.read_file(ROOFTOP_INVENTORY)
if rooftops.crs is None:
    rooftops = rooftops.set_crs(epsg=4326)
rooftops_wgs84 = rooftops.to_crs(epsg=4326)

area_column = "area" if "area" in rooftops.columns else None
inventory_summary = {
    "rooftop_count": int(len(rooftops)),
    "crs": str(rooftops.crs),
}
if area_column:
    inventory_summary.update({
        "total_rooftop_area_km2": float(rooftops[area_column].sum() / 1_000_000),
        "median_rooftop_area_m2": float(rooftops[area_column].median()),
        "p90_rooftop_area_m2": float(rooftops[area_column].quantile(0.90)),
    })

display(pd.DataFrame([inventory_summary]))

In [ ]:
preview = rooftops_wgs84.sample(min(len(rooftops_wgs84), 1200), random_state=7) if len(rooftops_wgs84) > 1200 else rooftops_wgs84
center = [float(preview.geometry.centroid.y.mean()), float(preview.geometry.centroid.x.mean())]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB Positron")
folium.GeoJson(
    preview.to_json(),
    name="Rooftop inventory sample",
    style_function=lambda feature: {
        "fillColor": "#2563eb",
        "color": "#1e3a8a",
        "weight": 0.4,
        "fillOpacity": 0.28,
    },
).add_to(m)
folium.LayerControl().add_to(m)
display(m)

if area_column:
    fig = px.histogram(
        rooftops,
        x=area_column,
        nbins=120,
        title="Rooftop area distribution before model execution",
        labels={area_column: "Rooftop area (m2)"},
    )
    fig.update_layout(height=360, bargap=0.04)
    fig.show()

## 2. Model Selection and Parameters

The model is selected because its declared inputs match the research design: a rooftop vector package, an efficiency assumption, and a start/end assessment period. This cell checks the local PyGeoModel catalog used by the runtime before a task is submitted.

In [ ]:
modeler = GeoModeler()
model = modeler.get_model(MODEL_NAME)

model_inputs = pd.DataFrame([item.to_dict() for item in model.inputs])
model_outputs = pd.DataFrame([item.to_dict() for item in model.outputs])

print(model.display_name or model.name)
print(model.model_id)
display(model_inputs[["name", "event_name", "data_type", "is_file", "required"]])
display(model_outputs[["name", "event_name", "data_type"]])

## 3. Submit the Live OpenGMS Task

This is the experimental run. The task may take several minutes because the service computes rooftop-level PV potential for the requested assessment period. The notebook passes the local rooftop package to PyGeoModel, waits for the service result, downloads the returned files, and records the task metadata in the same run directory.

If the runtime needs private credentials, configure OGMS_TOKEN in the server environment before launching Jupyter. The notebook does not contain or print any token.

In [ ]:
params = {
    "system_efficiency": 0.8,
    "start_time": 201801,
    "end_time": 201812,
    "roof_vector_path": str(MODEL_INPUT_ZIP),
}

configure_opengms_http_timeout()

print("Submitting live OpenGMS rooftop PV task...")
live_result = modeler.invoke(
    MODEL_NAME,
    params=params,
    output_dir=DOWNLOAD_DIR,
    record_path=RUN_DIR / "task_result.json",
)

if not live_result.downloaded_outputs:
    raise RuntimeError("The OpenGMS task completed without downloadable outputs; no historical output will be substituted.")

print(f"Task status: {live_result.status}")
print(f"Downloaded {len(live_result.downloaded_outputs)} output file(s).")
display(pd.DataFrame(live_result.outputs))
live_result.downloaded_outputs

## 4. Resolve the Returned Shapefile

OpenGMS returns one or more downloadable packages. The analysis below unpacks the files from the current task and locates the shapefile that contains rooftop-level PV attributes. This is the only result source used by the remaining cells.

In [ ]:
def unpack_downloaded_outputs(downloaded_outputs, extract_dir):
    extract_dir.mkdir(parents=True, exist_ok=True)
    discovered_files = []
    for item in downloaded_outputs:
        downloaded_path = Path(item)
        if not downloaded_path.exists():
            raise FileNotFoundError(f"Downloaded output was not found: {downloaded_path}")
        if downloaded_path.suffix.lower() == ".zip":
            target_dir = extract_dir / downloaded_path.stem
            target_dir.mkdir(parents=True, exist_ok=True)
            root = target_dir.resolve()
            with zipfile.ZipFile(downloaded_path) as archive:
                for member in archive.infolist():
                    member_target = (target_dir / member.filename).resolve()
                    if member_target != root and root not in member_target.parents:
                        raise RuntimeError(f"Blocked unexpected zip member path: {member.filename}")
                    archive.extract(member, target_dir)
            discovered_files.extend([path for path in target_dir.rglob("*") if path.is_file()])
        else:
            discovered_files.append(downloaded_path)
    return sorted(discovered_files)


def find_live_output(paths, suffix, preferred_terms=()):
    matches = [path for path in paths if path.suffix.lower() == suffix.lower()]
    for term in preferred_terms:
        preferred = [path for path in matches if term.lower() in path.name.lower()]
        if preferred:
            return sorted(preferred)[0]
    if matches:
        return sorted(matches)[0]
    available = ", ".join(str(path) for path in paths[:20])
    raise FileNotFoundError(f"No live output with suffix {suffix} was found. Available files: {available}")

live_files = unpack_downloaded_outputs(live_result.downloaded_outputs, EXTRACT_DIR)
live_result_shp = find_live_output(live_files, ".shp", preferred_terms=("power", "roof"))
print(f"Using live model result shapefile: {live_result_shp}")

pv_gdf = gpd.read_file(live_result_shp)
missing_fields = {"power_gene"}.difference(pv_gdf.columns)
if missing_fields:
    raise KeyError(f"The live model output is missing expected field(s): {sorted(missing_fields)}")

pv_gdf["power_gene"] = pd.to_numeric(pv_gdf["power_gene"], errors="coerce")
pv_gdf = pv_gdf.dropna(subset=["power_gene"]).copy()
if pv_gdf.empty:
    raise RuntimeError("The live model output contains no valid rooftop power generation values.")

if pv_gdf.crs is None:
    pv_gdf = pv_gdf.set_crs(epsg=4326)

pv_gdf.head()

## 5. Analyze the Live Model Result

The following statistics are calculated from the freshly downloaded rooftop result. They answer the research question in measurable terms: total annual generation potential, typical rooftop contribution, spatial concentration, and carbon reduction under a stated grid-emission assumption.

In [ ]:
generation = pv_gdf["power_gene"].clip(lower=0)
area_result_column = "area" if "area" in pv_gdf.columns else None
installed_column = next((name for name in ["installed_", "installed", "capacity"] if name in pv_gdf.columns), None)

GRID_EMISSION_FACTOR_KG_PER_KWH = 0.5
analysis_summary = {
    "run_id": RUN_ID,
    "result_shapefile": str(live_result_shp),
    "rooftops_analyzed": int(len(pv_gdf)),
    "total_generation_kwh_per_year": float(generation.sum()),
    "total_generation_gwh_per_year": float(generation.sum() / 1_000_000),
    "median_generation_kwh_per_year": float(generation.median()),
    "p90_generation_kwh_per_year": float(generation.quantile(0.90)),
    "max_generation_kwh_per_year": float(generation.max()),
    "co2_reduction_tons_per_year": float(generation.sum() * GRID_EMISSION_FACTOR_KG_PER_KWH / 1000),
}
if area_result_column:
    analysis_summary["result_total_rooftop_area_km2"] = float(pd.to_numeric(pv_gdf[area_result_column], errors="coerce").sum() / 1_000_000)
if installed_column:
    analysis_summary["total_installed_capacity_field_sum"] = float(pd.to_numeric(pv_gdf[installed_column], errors="coerce").sum())

display(pd.DataFrame([analysis_summary]).T.rename(columns={0: "value"}))

In [ ]:
pv_wgs84 = pv_gdf.to_crs(epsg=4326)
pv_utm = pv_wgs84.to_crs(epsg=32650)
centroids = pv_utm.geometry.centroid.to_crs(epsg=4326)
pv_wgs84["latitude"] = centroids.y
pv_wgs84["longitude"] = centroids.x

weights = np.log1p(pv_wgs84["power_gene"].clip(lower=0))
if float(weights.max()) == float(weights.min()):
    normalized_weights = np.ones(len(weights))
else:
    normalized_weights = (weights - weights.min()) / (weights.max() - weights.min())

heat_data = [
    [float(row.latitude), float(row.longitude), float(normalized_weights.iloc[index])]
    for index, row in pv_wgs84.reset_index(drop=True).iterrows()
]

center = [float(pv_wgs84["latitude"].mean()), float(pv_wgs84["longitude"].mean())]
heatmap = folium.Map(location=center, zoom_start=13, tiles="CartoDB Positron")
HeatMap(
    heat_data,
    radius=6,
    blur=5,
    min_opacity=0.25,
    max_opacity=0.9,
    gradient={"0.0": "#1d4ed8", "0.35": "#22c55e", "0.7": "#facc15", "1.0": "#ef4444"},
).add_to(heatmap)
display(heatmap)

In [ ]:
summary_path = RUN_DIR / "analysis_summary.json"
summary_path.write_text(json.dumps(analysis_summary, indent=2), encoding="utf-8")
print(f"Saved analysis summary: {summary_path}")

## 6. Interpretation

The table and heatmap above are tied to the current OpenGMS task record in the run directory. If the model service, solar-radiation data, or rooftop input package changes, a later run can produce a different but traceable result. That is the intended behavior for this case: the notebook is a live computational experiment, not a static reproduction of a bundled output file.